# 01 — Preparazione Calibration Dataset

Notebook per costruire i 3 calibration dataset (**Random / Code / Mixed**) usati poi da `llama-imatrix`.

**Fonti** :
- **Random** → `allenai/c4` (config `en`)
- **Code** → `codeparrot/codeparrot-clean-valid`
- **Mixed** → 50% `codeparrot-clean-valid` + 50% `HuggingFaceH4/stack-exchange-preferences` (filtrato su "python")


## 1. Setup

### Login su Hugging Face

In [1]:
import os
from huggingface_hub import login

with open(os.path.expanduser("~/.cache/huggingface/token")) as f:
    token = f.read().strip()

login(token=token)
print("Login effettuato nel notebook.")

Login effettuato nel notebook.


## 2. Caricamento dataset in streaming

Carichiamo le fonti per **Random** e **Code**

In [5]:
from datasets import load_dataset

# Random: corpus generico da web crawl
ds_random = load_dataset("allenai/c4", "en", split="train", streaming=True)

# Code: file Python puliti da GitHub
ds_code = load_dataset("codeparrot/codeparrot-clean-valid", split="train", streaming=True)

print("Dataset caricati in streaming.")

README.md:   0%|          | 0.00/41.1k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/401 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


Dataset caricati in streaming.


### Ispezione struttura dati

Controlliamo le chiavi disponibili per capire quali campi usare per estrarre il testo.

In [6]:
example_random = next(iter(ds_random))
print("RANDOM keys:", example_random.keys())

example_code = next(iter(ds_code))
print("CODE keys:", example_code.keys())

RANDOM keys: dict_keys(['text', 'timestamp', 'url'])
CODE keys: dict_keys(['repo_name', 'path', 'copies', 'size', 'content', 'license', 'hash', 'line_mean', 'line_max', 'alpha_frac', 'autogenerated'])


## 3. Aggiunta fonte StackExchange (per il dataset Mixed)

In [7]:
ds_stackexchange = load_dataset(
    "HuggingFaceH4/stack-exchange-preferences",
    split="train",
    streaming=True,
)

example_se = next(iter(ds_stackexchange))
print("STACKEXCHANGE keys:", example_se.keys())
print(example_se)

README.md:   0%|          | 0.00/8.76k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/758 [00:00<?, ?it/s]

STACKEXCHANGE keys: dict_keys(['qid', 'question', 'answers', 'date', 'metadata'])
{'qid': 1, 'question': '<p>I have been wanting to learn about 3D printing a long time so I really want this site to succeed but I have no previous experience with the subject. </p>\n\n<p>I was wondering how can I help the site at this early stage. I thought about asking about how to get started with 3D printing but SE explicitly discourages "easy" questions in the private beta.</p>\n\n<p>What can newbies like me do for the site at this stage besides voting questions and answers?</p>\n', 'answers': [{'answer_id': 14, 'author': 'Eric Johnson', 'author_id': 43, 'author_profile': 'https://3dprinting.meta.stackexchange.com/users/43', 'pm_score': 2, 'selected': False, 'text': '<p>I would suggest doing a bit of basic research on 3D printing (including reading questions and answers).  From these you will learn more about it and hopefull you will have new questions about 3D printing that can be asked.  </p>\n\n<p>

In [8]:
ds_stackoverflow = load_dataset(
    "HuggingFaceH4/stack-exchange-preferences",
    split="train",
    data_dir="data/Stackoverflow.com",
    streaming=True,
)

example_so = next(iter(ds_stackoverflow))
print(example_so["question"][:300])

Resolving data files:   0%|          | 0/335 [00:00<?, ?it/s]

<p>I want to assign the decimal variable &quot;trans&quot; to the double variable &quot;this.Opacity&quot;.</p>
<pre class="lang-cs prettyprint-override"><code>decimal trans = trackBar1.Value / 5000;
this.Opacity = trans;
</code></pre>
<p>When I build the app it gives the following error:</p>
<block


### Filtro sulle domande relative a Python

In [9]:
def is_python_related(example):
    text = example["question"].lower()
    return "python" in text

ds_stackoverflow_python = filter(is_python_related, ds_stackoverflow)

# Verifica veloce: prendi i primi 2 esempi filtrati
for i, ex in enumerate(ds_stackoverflow_python):
    print(f"--- Esempio {i+1} ---")
    print(ex["question"][:300])
    print()
    if i >= 1:
        break

--- Esempio 1 ---
<p>I am about to build a piece of a project that will need to construct and post an XML document to a web service and I'd like to do it in Python, as a means to expand my skills in it.  </p>

<p>Unfortunately, whilst I know the XML model fairly well in .NET, I'm uncertain what the pros and cons are 

--- Esempio 2 ---
<p>I am using the Photoshop's javascript API to find the fonts in a given PSD.</p>

<p>Given a font name returned by the API, I want to find the actual physical font file that font name corresponds to on the disc.</p>

<p>This is all happening in a python program running on OSX so I guess I'm lookin



## 4. Costruzione dei tre calibration dataset

Costruiamo `random.txt`, `code.txt`, `mixed.txt` (~300 campioni ciascuno).

Per lo StackExchange rimuoviamo i tag HTML per ottenere testo pulito.

### Funzioni di supporto

In [10]:
import re
import html

def strip_html(raw_html: str) -> str:
    text = re.sub(r"<[^>]+>", " ", raw_html)
    text = html.unescape(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
N_SAMPLES = 300
MAX_CHARS_PER_SAMPLE = 3000  # evita campioni eccessivamente lunghi

def build_txt_file(iterator, extract_fn, n_samples, output_path):
    count = 0
    with open(output_path, "w", encoding="utf-8") as f:
        for example in iterator:
            text = extract_fn(example)
            if not text or len(text.strip()) < 50:
                continue
            text = text[:MAX_CHARS_PER_SAMPLE]
            f.write(text + "\n\n---\n\n")
            count += 1
            if count >= n_samples:
                break
    print(f"Scritti {count} campioni in {output_path}")

Scritti 300 campioni in ../calibration_data/random.txt


### 4.1 — `random.txt` (100% C4, testo generico)

In [15]:
build_txt_file(
    ds_random,
    extract_fn=lambda ex: ex["text"],
    n_samples=N_SAMPLES,
    output_path="../calibration_data/random.txt",
)

Scritti 300 campioni in ../calibration_data/random.txt


### 4.2 — `code.txt` (100% GitHub Python)

In [12]:
build_txt_file(
    ds_code,
    extract_fn=lambda ex: ex["content"],
    n_samples=N_SAMPLES,
    output_path="../calibration_data/code.txt",
)

Scritti 300 campioni in ../calibration_data/code.txt


### 4.3 — `mixed.txt` (50% GitHub Python + 50% StackExchange Python)

In [ ]:
N_HALF = N_SAMPLES // 2  # 150 + 150 = 300

count = 0
with open("../calibration_data/mixed.txt", "w", encoding="utf-8") as f:
    # metà 1: codice GitHub Python 
    ds_code_fresh = load_dataset("codeparrot/codeparrot-clean-valid", split="train", streaming=True)
    for example in ds_code_fresh:
        text = example["content"]
        if not text or len(text.strip()) < 50:
            continue
        f.write(text[:MAX_CHARS_PER_SAMPLE] + "\n\n---\n\n")
        count += 1
        if count >= N_HALF:
            break

    # metà 2: StackExchange Python (già filtrato)
    count_se = 0
    for example in ds_stackoverflow_python:
        text = strip_html(example["question"])
        if not text or len(text.strip()) < 50:
            continue
        f.write(text[:MAX_CHARS_PER_SAMPLE] + "\n\n---\n\n")
        count += 1
        count_se += 1
        if count_se >= N_HALF:
            break

print(f"Scritti {count} campioni totali in ../calibration_data/mixed.txt ({N_HALF} codice + {count_se} StackExchange)")

Repo card metadata block was not found. Setting CardData to empty.


Scritti 300 campioni totali in ../calibration_data/mixed.txt (150 codice + 150 StackExchange)


## 5. Verifica finale dei tre file

In [14]:
import os

for fname in ["random.txt", "mixed.txt", "code.txt"]:
    path = f"../calibration_data/{fname}"
    size_kb = os.path.getsize(path) / 1024
    with open(path, "r", encoding="utf-8") as f:
        n_lines = sum(1 for _ in f)
    print(f"{fname}: {size_kb:.1f} KB, {n_lines} righe")

random.txt: 429.3 KB, 2781 righe
mixed.txt: 463.0 KB, 12375 righe
code.txt: 726.3 KB, 23396 righe
